In [1]:
# Core RAG Frameworks
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-text-splitters
!pip install langchain-experimental
!pip install langgraph


In [2]:
import json
import os

from openai import OpenAI
from google.colab import userdata

from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker  # AI-powered semantic chunking
from langchain_community.vectorstores import Chroma  # Vector database
from langchain_openai import OpenAIEmbeddings  # OpenAI embedding function
from langchain_core.documents import Document  # Document abstraction


from langchain_text_splitters import (  # Various splitting strategies
    RecursiveCharacterTextSplitter,  # Best general-purpose splitter
    CharacterTextSplitter,  # Simple split by character count
    MarkdownHeaderTextSplitter,  # Splits based on markdown headers
    HTMLHeaderTextSplitter  # Splits based on HTML tags
)


/tmp/ipykernel_3192/929496560.py:8: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker  # AI-powered semantic chunking


In [3]:
# =============================================================================
# LOAD DATA: Support Tickets
# =============================================================================
print("\nLoading support tickets...")
with open('./sample_data/synthetic_tickets.json', 'r') as f:
    tickets = json.load(f)
print(f"Loaded {len(tickets)} support tickets")


Loading support tickets...
Loaded 22 support tickets


In [4]:
# Display sample ticket - understand what we're working with
print("\n" + "="*80)
print("SAMPLE TICKET:")
print("="*80)
sample = tickets[0]
print(f"ID: {sample['ticket_id']}")
print(f"Title: {sample['title']}")
print(f"Description: {sample['description'][:200]}...")
print(f"Category: {sample['category']}")
print(f"Priority: {sample['priority']}")

print("\n" + "="*80)
print("PART 1: Chunking Strategies")
print("="*80)


SAMPLE TICKET:
ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment....
Category: Authentication
Priority: High

PART 1: Chunking Strategies


In [5]:
# -----------------------------------------------------------------------------
# First, convert our ticket data into LangChain Document objects
# Document = page_content (text) + metadata (structured info for filtering)
# -----------------------------------------------------------------------------

documents = []
for ticket in tickets:
    # Combine all ticket fields into a single text block
    # TIP: Include all relevant context that helps understand the document
    full_text = f"""
Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Category: {ticket['category']}
Priority: {ticket['priority']}
Description: {ticket['description']}
Resolution: {ticket['resolution']}
    """.strip()


    # Create Document object with metadata
    # Metadata is CRUCIAL - it enables filtering later!
    # Example: "Find similar tickets, but only in the 'Authentication' category"
    doc = Document(
        page_content=full_text,  # The actual text content
        metadata={
            'ticket_id': ticket['ticket_id'],   # For identifying results
            'category': ticket['category'],      # For category filtering
            'description': ticket['description'],      # For descriptopn filtering
            'resolution': ticket['resolution']       # For resolution filtering
        }
    )
    documents.append(doc)

print(f"Created {len(documents)} documents")
print(f"\nSample document length: {len(documents[0].page_content)} characters")

Created 22 documents

Sample document length: 523 characters


In [6]:
# Vector Stores
!pip install chromadb
!pip install langchain-chroma

In [7]:
# ============================================================================
# PART 2: Chroma Vector Store
# ============================================================================

print("\n" + "="*80)
print("PART 2: Chroma Vector Store")
print("="*80)

# Use LangChain's embedding wrapper (handles API calls internally)
embeddings_model = OpenAIEmbeddings(
    model='text-embedding-3-small',
    api_key=userdata.get('OPENAI_APIKEY')
)



print("\nBuilding Chroma vector store...")

# Clean up any existing collection to avoid duplicates on re-run
# from_documents() APPENDS to existing collections, so without this
# you'd get duplicate documents (and duplicate search results!) each run
existing_store = Chroma(
    collection_name="support_tickets",
    persist_directory="./chroma_db"
)
existing_store.delete_collection()

# from_documents() handles everything:
#   1. Extracts text from each Document
#   2. Generates embeddings via the embedding model
#   3. Stores vectors + metadata + original text
#   4. Persists to disk (if persist_directory specified)
chroma_store = Chroma.from_documents(
    documents=documents,              # Our LangChain Document objects
    embedding=embeddings_model,       # OpenAI embeddings
    collection_name="support_tickets",# Like a "table" in a database
    persist_directory="./chroma_db"   # Save to disk for persistence
)
print("✓ Chroma store created and persisted")


PART 2: Chroma Vector Store

Building Chroma vector store...


/tmp/ipykernel_3192/3778981295.py:22: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  existing_store = Chroma(


✓ Chroma store created and persisted


In [8]:
# -----------------------------------------------------------------------------
# Basic Similarity Search with query with score with filter on Authentication
# -----------------------------------------------------------------------------

query1 = "Authentication problems after password reset"

print(f"\nSearching in Chroma: '{query1}' with Authentication Filter")
chroma_results1 = chroma_store.similarity_search_with_score(query1, k=5, filter={"category": "Authentication"})

print(f"\nTop {len(chroma_results1)} results:")
for i, (doc1,score1) in enumerate(chroma_results1, 1):
    print(f"\n#{i} - Distance: {score1:.4f}")
    print(f"Ticket: {doc1.metadata['ticket_id']}")
    print(f"Category: {doc1.metadata['category']}")
    print(f"Description: {doc1.metadata['description']}")
    print(f"Resolution: {doc1.metadata['resolution']}")





Searching in Chroma: 'Authentication problems after password reset' with Authentication Filter

Top 4 results:

#1 - Distance: 0.8060
Ticket: TICK-001
Category: Authentication
Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.
Resolution: Found that the password hash algorithm was updated but session tokens weren't invalidated. Solution: Clear all active sessions and force re-authentication. Implemented automatic session cleanup on password change.

#2 - Distance: 1.1431
Ticket: TICK-016
Category: Authentication
Description: Users unable to log in with TOTP codes from authenticator apps. Codes shown as invalid even when entered immediately after generation. Affects random subset of users.
Resolution: Server time was drifting by 2 minutes due to misconfigured NTP. TOTP codes are time-sensitive and were outside acceptance window. Configured NTP synchron

In [9]:
# -----------------------------------------------------------------------------
# Basic Similarity Search with query with filter on Database
# -----------------------------------------------------------------------------

query2 = "Connection problems after restart"

print(f"\nSearching in Chroma: '{query2}' with Database Filter")
chroma_results2 = chroma_store.similarity_search_with_score(query2, k=5, filter={"category": "Database"})

print(f"\nTop {len(chroma_results2)} results:")
for i, (doc2,score2) in enumerate(chroma_results2, 1):
    print(f"\n#{i} - Distance: {score2:.4f}")
    print(f"Ticket: {doc2.metadata['ticket_id']}")
    print(f"Category: {doc2.metadata['category']}")
    print(f"Description: {doc2.metadata['description']}")
    print(f"Resolution: {doc2.metadata['resolution']}")




Searching in Chroma: 'Connection problems after restart' with Database Filter

Top 2 results:

#1 - Distance: 1.3264
Ticket: TICK-002
Category: Database
Description: Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.
Resolution: Database connection pool was sized too small for peak load. Increased max_connections from 20 to 100. Added connection pooling monitoring and alerts. Optimized long-running queries that were holding connections.

#2 - Distance: 1.3288
Ticket: TICK-021
Category: Database
Description: Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.
Resolution: Database connection pool was sized too big for peak load. Increased min_connections from 20 to 10. Added connection pooling monitoring and alerts. Terminating 

In [10]:
# -----------------------------------------------------------------------------
# Basic Similarity Search with query with filter on Performance
# -----------------------------------------------------------------------------

query3 = "User experience slow loading"

print(f"\nSearching in Chroma: '{query3}' with Performace Filter")
chroma_results3 = chroma_store.similarity_search_with_score(query3, k=5, filter={"category": "Performance"})

print(f"\nTop {len(chroma_results3)} results:")
for i, (doc3,score3) in enumerate(chroma_results3, 1):
    print(f"\n#{i} - Distance: {score3:.4f}")
    print(f"Ticket: {doc3.metadata['ticket_id']}")
    print(f"Category: {doc3.metadata['category']}")
    print(f"Description: {doc3.metadata['description']}")
    print(f"Resolution: {doc3.metadata['resolution']}")




Searching in Chroma: 'User experience slow loading' with Performace Filter

Top 2 results:

#1 - Distance: 0.8581
Ticket: TICK-010
Category: Performance
Description: Admin dashboard takes 30+ seconds to load. Browser DevTools show multiple slow database queries. Page becomes unresponsive during load. Affects all users with admin privileges.
Resolution: Dashboard was making N+1 queries for user statistics. Implemented query optimization using JOIN statements and added database indexes on frequently queried columns. Introduced Redis caching for aggregate statistics with 5-minute TTL. Load time reduced to under 2 seconds.

#2 - Distance: 1.3887
Ticket: TICK-005
Category: Performance
Description: Background job processor showing steady memory increase over time. Process killed by OS after reaching 8GB memory usage. Requires daily restart to maintain functionality.
Resolution: Memory profiling revealed unclosed database cursors in batch processing loop. Added explicit cursor.close() calls 

In [11]:
# -----------------------------------------------------------------------------
# Basic Similarity Search with query with filter on Email
# -----------------------------------------------------------------------------

query4 = "User registeration issue"

print(f"\nSearching in Chroma: '{query4}' with Email Filter")
chroma_results4 = chroma_store.similarity_search_with_score(query4, k=5, filter={"category": "Email"})

print(f"\nTop {len(chroma_results4)} results:")
for i, (doc4,score4) in enumerate(chroma_results4, 1):
    print(f"\n#{i} - Distance: {score4:.4f}")
    print(f"Ticket: {doc4.metadata['ticket_id']}")
    print(f"Category: {doc4.metadata['category']}")
    print(f"Description: {doc4.metadata['description']}")
    print(f"Resolution: {doc4.metadata['resolution']}")




Searching in Chroma: 'User registeration issue' with Email Filter

Top 2 results:

#1 - Distance: 0.8904
Ticket: TICK-022
Category: Email
Description: Users complaining that email used for registeration is failing.
Resolution: Email service provider had aalready registered . To reregister they need delete the account and then register

#2 - Distance: 1.4033
Ticket: TICK-006
Category: Email
Description: Users reporting they're not receiving password reset and confirmation emails. SMTP server logs show successful sends, but emails never arrive in inbox or spam folder.
Resolution: Email service provider had blacklisted our sending IP due to spam complaints. Implemented SPF, DKIM, and DMARC records. Switched to dedicated IP address with proper warmup process. Added email deliverability monitoring.
